# 03 - Improvement: a learned encoder for the projection step

**Idea.** The paper projects an embedding by *searching*: draw k random latents z, generate k candidates, keep the closest. That search is crude: with k=30 in a 20-40 dimensional latent space the best sample is rarely the best latent, and the random draws make the defense stochastic. Instead, train an **encoder** `q(z | t, c)` that predicts the latent directly (amortised GAN inversion), so projection becomes one deterministic pass: `t_hat = G(mu(t, c), c)`.

**How the encoder is trained** (generator frozen):
- reconstruction `||G(z, c) - t_clean||`, with a small KL term to keep latents near the prior (so the generator is queried where it was trained);
- a cycle term `||mu(G(z0, c0), c0) - z0||^2` on synthetic samples, which anchors the encoder to the generator's own manifold;
- **denoising**: the encoder input is the clean embedding plus gaussian noise, and half the time the embedding of a *synonym-perturbed* copy of the same training text (counter-fitted neighbours from `TextDefender/counterfitted_neighbors.json`), while the target stays the clean embedding. So it learns to map perturbed embeddings back to the natural point, which is exactly what a defense needs.

**Variants run here:** `encoder` (deterministic, one pass, cheaper than the paper's search) and `hybrid` (encoder mean plus k draws around it; keep the closest, so it can only improve the reconstruction over the encoder alone).

**Fairness note:** the denoising variant uses synonym perturbations during *training* of the projector. The attacks use the same counter-fitted neighbours for PWWS/TextFooler, so a gain on those two attacks is partly 'trained on similar perturbations'. BERT-Attack uses BERT masked-LM substitutions the encoder never sees, so it is the cleaner generalisation check. Set `USE_AUG = False` for the stricter version (noise only).

Run notebook 00 first.

In [ ]:
!pip -q install transformers datasets accelerate textattack tensorflow_hub
import os, sys, subprocess
REPO = 'https://github.com/shivanshgoyaliitk/for_shreyansh.git'
BRANCH = 'claude/magical-pasteur-ptp2k2'
DIR = '/content/for_shreyansh'
url = REPO
try:  # only needed if the repo is private: add a Colab secret named GITHUB_TOKEN
    from google.colab import userdata
    url = REPO.replace('https://', 'https://' + userdata.get('GITHUB_TOKEN') + '@')
except Exception:
    pass
if os.path.exists(DIR):
    subprocess.run(['git', '-C', DIR, 'pull', '-q'])
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, DIR], check=True)
sys.path.insert(0, DIR)

In [ ]:
import torch, tmd_lab as T
T.setup_env()
from google.colab import drive
drive.mount('/content/drive')

DATASET = 'ag_news'   # must match notebook 00
N_EVAL = 1000
BASE_K = 30
WORK = '/content/drive/MyDrive/tmd_run'
USE_AUG = True        # denoising with synonym-perturbed embeddings (see note above)
ENC_EPOCHS = 30
VARIANTS = ['encoder', 'hybrid']   # evaluated with the full attack suite; drop one to save time

lab = T.Lab(DATASET, work=WORK)
lab.finetune('base')
E_tr, Y_tr = lab.embeddings('train', 'base')
E_va, Y_va = lab.embeddings('val', 'base')
E_te, Y_te = lab.embeddings('test', 'base')
gan = lab.get_gan('base', E_tr, E_va)

base = T.Victim(lab, gan=gan, mode='tmd', proj_mode='sample', k=BASE_K)
T.evaluate(lab, base, 'baseline', E_te, Y_te, n=N_EVAL)   # cached if notebook 00 already ran

In [ ]:
E_aug = None
if USE_AUG:
    aug = T.SynonymAugmenter(f'{DIR}/TextDefender/counterfitted_neighbors.json', rate=lab.cfg['rho'], seed=0)
    E_aug, _ = lab.embeddings('train', 'base', augment_fn=aug, name='train_aug')

ENC_NAME = 'denoise' if USE_AUG else 'noise'
enc = lab.get_encoder(ENC_NAME, gan, E_tr, E_va, epochs=ENC_EPOCHS, E_aug=E_aug)

In [ ]:
# Cheap gate before the expensive attacks: reconstruction quality and clean accuracy on the full test set.
Et = E_te.to(T.DEV)
for name, kw in [('paper sampling (k=%d)' % BASE_K, dict(mode='sample', k=BASE_K)),
                 ('encoder', dict(mode='encoder')),
                 ('hybrid', dict(mode='hybrid', k=BASE_K))]:
    th, d = T.project(gan, Et, enc=enc, **kw)
    acc = (lab.model.classifier(th).argmax(-1).cpu() == Y_te).float().mean().item() * 100
    print(f'{name:24s} mean ||t_hat - t|| = {d.mean().item():.3f}   clean acc = {acc:.2f}')
print('baseline clean acc in our run:', round(T.json.load(open(lab._path('results_baseline.json')))['clean']['acc'], 2))
print('If the encoder reconstruction loss is much worse than sampling, train longer or raise ENC_EPOCHS before attacking.')

In [ ]:
import pandas as pd
tables = []
for var in VARIANTS:
    tag = f'enc_{var}_{ENC_NAME}'
    v = T.Victim(lab, gan=gan, enc=enc, mode='tmd', proj_mode=var, k=BASE_K)
    T.evaluate(lab, v, tag, E_te, Y_te, n=N_EVAL)
    print('\n==', tag)
    display(T.compare(lab, tag, baseline_tag='baseline'))